In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
dbutils.widgets.text("nameContainer","unit-catalog")
dbutils.widgets.text("nameStorage","adlsmovieproject")
dbutils.widgets.text("catalogo","catalog_movie_project")


In [0]:
%python

nameContainer = dbutils.widgets.get("nameContainer")
nameStorage = dbutils.widgets.get("nameStorage")
catalogo = dbutils.widgets.get("catalogo")

storageLocation = f"abfss://{nameContainer}@{nameStorage}.dfs.core.windows.net"
## dbutils.widgets.text("storageLocation",ruta)
## storageLocation1=dbutils.widgets.get("storageLocation")

In [0]:

%sql
use catalog ${catalogo}

### Creacion Tablas BRONZE

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.bronze.MOREINFO_PP (
 id       INT
,runtime  STRING
,budget   STRING
,revenue  STRING
,poster_path   STRING
,backdrop_path STRING
)
USING DELTA
LOCATION '{storageLocation}/bronze/MOREINFO_PP'
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.bronze.MOVIES (
 id            INT,
 title         STRING,
 genres        STRING,
 language      STRING,
 user_score    DOUBLE,
 runtime_hour  INT,
 runtime_min   INT,
 release_date  DATE,
 vote_count    INT
)
USING DELTA
LOCATION '{storageLocation}/bronze/MOVIES'
""")

### Creacion tablas SILVER

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.silver.MOVIES_CLEAN (
 id            INT,
 title         STRING,
 genres        ARRAY<STRING>,
 language      STRING,
 user_score    DOUBLE,
 runtime_hour  INT,
 runtime_min   INT,
 release_date  DATE,
 vote_count    INT
)
USING DELTA
LOCATION '{storageLocation}/silver/MOVIES_CLEAN'
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.silver.FILMDETAILS_CLEAN (
 id           INT 
,director     STRING
,top_billed   STRING
,budget_usd   DOUBLE
,revenue_usd  DOUBLE
)
USING DELTA
LOCATION '{storageLocation}/silver/FILMDETAILS_CLEAN'
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.silver.MOREINFO_CLEAN (
 id       INT
,runtime  STRING
,budget   DOUBLE
,revenue  DOUBLE
,poster_path   STRING
,backdrop_path STRING
)
USING DELTA
LOCATION '{storageLocation}/silver/MOREINFO_CLEAN'
""")

### Creacion tablas Golden

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.golden.MOVIE_MASTER (
id           INT
,title        STRING
,genres       ARRAY<STRING>
,language     STRING
,director     STRING
,top_billed   STRING
,runtime_min  INT
,release_year DATE
,user_score   DOUBLE
,vote_count   INT
,budget_usd   DOUBLE
,revenue_usd  DOUBLE
,profit       DOUBLE
,roi          DOUBLE
)
USING DELTA
LOCATION '{storageLocation}/golden/MOVIE_MASTER'
""")